# Rusanescu Andrei-Marian 333CC Tema1

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import json
import warnings
import importlib.util

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

RNG = 42
ROOT = Path.cwd()
TRAIN_PATH = os.path.join(os.path.dirname(ROOT), "CC_education_economy_train.csv")
TEST_PATH  = os.path.join(os.path.dirname(ROOT), "CC_education_economy_test.csv")

FIG = os.path.join(ROOT, "figures")
RES = os.path.join(ROOT, "results")
os.makedirs(FIG, exist_ok=True)
os.makedirs(RES, exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG, name), bbox_inches="tight")
    plt.close()
    print("[fig]", name)

def cramers_v(a, b):
    """Cramer's V"""
    ct = pd.crosstab(a, b)
    chi2 = chi2_contingency(ct)[0]
    n = ct.to_numpy().sum()
    r, k = ct.shape
    phi2 = chi2 / n
    phi2c = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1)
    kc = k - (k - 1) ** 2 / (n - 1)
    denom = min(kc - 1, rc - 1)
    return float(np.sqrt(phi2c / denom)) if denom > 0 else 0.0

def corr_ratio(cat, num):
    """eta - cat explica cat de mult din variatia num (categoric - numeric)."""
    df = pd.DataFrame({"c": cat, "n": num}).dropna()
    ybar = df["n"].mean()
    ss_tot = ((df["n"] - ybar) ** 2).sum()
    if ss_tot == 0:
        return 0.0
    ss_b = 0.0
    for c in df["c"].unique():
        sub = df.loc[df["c"] == c, "n"]
        ss_b += len(sub) * (sub.mean() - ybar) ** 2
    return float(np.sqrt(ss_b / ss_tot))

In [4]:
# Load data
df = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)
print("Train shape:", df.shape)

numeric = ["experience_years", "skills_count", "certifications",
           "total_days_worked", "aggregated_score", "salary"]
categ = ["job_title", "education_level", "industry", "company_size",
         "location", "remote_work", "skill_bracket"]

Train shape: (64000, 14)


# CERINTA 1 - EDA

In [5]:
# 1. Statistici numerice
num_stats = df[numeric].describe().T
num_stats["non_null"] = df[numeric].notna().sum()
num_stats.to_csv(os.path.join(RES, "eda_numeric_stats.csv"))
print("\nStatistici numerice:")
print(num_stats.round(2))


Statistici numerice:
                     count       mean       std      min        25%       50%  \
experience_years   64000.0       9.97      6.05      0.0       5.00      10.0   
skills_count       64000.0      11.60     10.74      1.0       5.00      10.0   
certifications     64000.0       2.51      1.71      0.0       1.00       3.0   
total_days_worked  64000.0    2393.51   1452.04     -1.0    1200.00    2400.0   
aggregated_score   64000.0       0.00      1.00     -4.8      -0.67       0.0   
salary             64000.0  145683.57  37384.76  31867.0  119209.75  143299.0   

                         75%        max  non_null  
experience_years       15.00      20.00     64000  
skills_count           15.00      69.00     64000  
certifications          4.00       5.00     64000  
total_days_worked    3600.00    4800.00     64000  
aggregated_score        0.68       4.22     64000  
salary             169582.25  333046.00     64000  


In [6]:
# 2. Statistici categorice
cat_rows = []
for c in categ + ["vacation"]:
    cat_rows.append({
        "column": c,
        "non_null": int(df[c].notna().sum()),
        "missing": int(df[c].isna().sum()),
        "unique": int(df[c].nunique(dropna=True)),
        "top": df[c].mode(dropna=True).iloc[0],
        "top_freq": int(df[c].value_counts(dropna=True).iloc[0]),
    })

cat_stats = pd.DataFrame(cat_rows).set_index("column")
cat_stats.to_csv(os.path.join(RES, "eda_categorical_stats.csv"))
print("\nStatistici categorice:")
print(cat_stats)


Statistici categorice:
                 non_null  missing  unique          top  top_freq
column                                                           
job_title           64000        0      12  AI Engineer      5410
education_level     64000        0       5     Bachelor     12863
industry            64000        0      10   Consulting      6547
company_size        64000        0       5        Large     12934
location            64000        0      10       Sweden      6570
remote_work         44847    19153       3       Hybrid     15054
skill_bracket       64000        0       3          low     21470
vacation            64000        0       4  No Vacation     34686


In [7]:
print("Distribuția pe clase pentru 'vacation':")
print(df['vacation'].value_counts())

print(df['vacation'].value_counts(normalize=True) * 100)

Distribuția pe clase pentru 'vacation':
vacation
No Vacation    34686
Small          12877
Medium          8527
Large           7910
Name: count, dtype: int64
vacation
No Vacation    54.196875
Small          20.120313
Medium         13.323437
Large          12.359375
Name: proportion, dtype: float64


In [8]:
# Fig 1 - valori lipsa
plt.figure(figsize=(9, 4))
miss = df.isna().mean().sort_values(ascending=False) * 100
miss.plot(kind="bar", color="tomato")
plt.title("Fig. 1 - Procent valori lipsa")
plt.ylabel("% valori lipsa")
plt.xticks(rotation=40, ha="right")

save("01_missing_values.png")


[fig] 01_missing_values.png


In [9]:
# Fig 2 - boxplot atr numerice
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), numeric):
    sns.boxplot(y=df[col], ax=ax, color="#4c72b0")
    ax.set_title(col)
plt.suptitle("Fig. 2 - Boxplot atribute numerice")

save("02_boxplots_numeric.png")

[fig] 02_boxplots_numeric.png


In [10]:
# Fig 3 - histograme numerice
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), numeric):
    sns.histplot(df[col], bins=40, kde=True, ax=ax, color="#55a868")
    ax.set_title(col)
    if col == "salary":
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f'{int(x/1000)}K'))
plt.suptitle("Fig. 3 - Distributia atributelor numerice")
save("03_hist_numeric.png")

[fig] 03_hist_numeric.png


In [11]:
# Fig 4 - histograme categorice
df_plot = df.copy()
df_plot["remote_work"] = df_plot["remote_work"].fillna("Missing")

# am pus width mai mare pentru ca nu incapeau numele
fig, axes = plt.subplots(3, 3, figsize=(18, 15))

for ax, col in zip(axes.ravel(), categ):
    order = df_plot[col].value_counts().index.tolist()
    sns.countplot(x=col, data=df_plot, order=order, ax=ax, color="#c44e52")

    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("count")

    ax.tick_params(axis="x", rotation=40)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment('right')

for ax in axes.ravel()[len(categ):]:
    ax.axis("off")

plt.subplots_adjust(hspace=0.6, wspace=0.3)
plt.suptitle("Fig. 4 - Distributia atributelor categorice")
save("04_hist_categorical.png")

[fig] 04_hist_categorical.png


In [12]:
# Fig 5 - dezechilibrul tintei vacation
def get_frequency_vacation(df_input, df_type):
    plt.figure(figsize=(7, 4))
    order = ["No Vacation", "Small", "Medium", "Large"]
    ax = sns.countplot(x="vacation", data=df_input, order=order, hue="vacation", palette="viridis", legend=False)
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(f"{h} ({100 * h / len(df):.1f}%)",
                    (p.get_x() + p.get_width() / 2, h),
                    ha="center", va="bottom", fontsize=9)
    plt.title(f"Fig. 5 - Dezechilibrul de clase pentru 'vacation' pt {df_type}")
    plt.ylabel("numar exemple")
    save(f"05_class_balance_{df_type}.png")

get_frequency_vacation(df, "antrenare")
get_frequency_vacation(df_test, "validare")

[fig] 05_class_balance_antrenare.png
[fig] 05_class_balance_validare.png


In [13]:
# Fig 6 - corelatie Pearson numerice
corr = df[numeric].corr(method="pearson")
corr.to_csv(os.path.join(RES, "correlation_numeric.csv"))
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Fig. 6 - Corelatie Pearson atribute numerice")
save("06_corr_numeric.png")

[fig] 06_corr_numeric.png


In [14]:
# Fig 7 - corelatie Cramer's V categorice
all_cat = categ + ["vacation"]
cv = pd.DataFrame(np.zeros((len(all_cat), len(all_cat))), index=all_cat, columns=all_cat)
for a in all_cat:
    for b in all_cat:
        cv.loc[a, b] = cramers_v(df[a].fillna("__NA__"), df[b].fillna("__NA__"))
cv.to_csv(os.path.join(RES, "correlation_categorical_cramersV.csv"))
plt.figure(figsize=(9, 7))
sns.heatmap(cv, annot=True, fmt=".2f", cmap="magma_r", square=True, vmin=0, vmax=1)
plt.title("Fig. 7 - Cramer's V perechi categorice (0 indep., 1 corelat)")
save("07_corr_categorical.png")

[fig] 07_corr_categorical.png


In [15]:

stats = df.groupby('skill_bracket')['skills_count'].agg(['min', 'max', 'count'])
print(stats.sort_values('min'))

               min  max  count
skill_bracket                 
low              1   69  21470
mid              7   69  21253
high            13   69  21277


Problema aici este ca outlierii (valorile mari din [60, 69]) apar in fiecare skill_bracket

In [16]:
# Fig 8 - corr_ratio numeric -> vacation
eta_cls = {c: corr_ratio(df["vacation"], df[c]) for c in numeric if c != "salary"}
eta_cls = pd.Series(eta_cls).sort_values(ascending=False)
plt.figure(figsize=(7, 4))
eta_cls.plot(kind="bar", color="#4c72b0")
plt.title("Fig. 8 - Corelatia atributelor numerice cu clasa 'vacation' (eta)")
plt.ylabel("correlation ratio")
plt.xticks(rotation=25, ha="right")
save("08_numeric_vs_classification.png")

[fig] 08_numeric_vs_classification.png


In [17]:
# Fig 9 - corelatii cu salary
pear = df[numeric].corr()["salary"].drop("salary").reindex(
    sorted(numeric, key=lambda c: abs(df[numeric].corr()["salary"].get(c, 0)), reverse=True)
).dropna()
eta_reg = {c: corr_ratio(df[c].fillna("__NA__"), df["salary"]) for c in categ}
eta_reg = pd.Series(eta_reg).sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pear.plot(kind="bar", ax=axes[0], color="#55a868")
axes[0].set_title("numeric -> salary (Pearson)")
axes[0].tick_params(axis="x", rotation=25)
eta_reg.plot(kind="bar", ax=axes[1], color="#c44e52")
axes[1].set_title("categoric -> salary (eta)")
axes[1].tick_params(axis="x", rotation=25)
plt.suptitle("Fig. 9 - Corelatia atributelor cu tinta de regresie 'salary'")
save("09_features_vs_salary.png")

[fig] 09_features_vs_salary.png


In [ ]:
# Fig 10 - salariu pe categorii cheie
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(x="vacation", y="salary", data=df,
            order=["No Vacation", "Small", "Medium", "Large"], ax=axes[0], palette="viridis")
axes[0].set_title("salary | vacation")
sns.boxplot(x="education_level", y="salary", data=df,
            order=["High School", "Diploma", "Bachelor", "Master", "PhD"],
            ax=axes[1], palette="Blues")
axes[1].set_title("salary | education_level")
sns.boxplot(x="company_size", y="salary", data=df,
            order=["Startup", "Small", "Medium", "Large", "Enterprise"],
            ax=axes[2], palette="Oranges")
axes[2].set_title("salary | company_size")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Fig. 10 - Salariu in functie de atribute cheie")
save("10_salary_by_cats.png")

In [ ]:
# Fig 11 - experienta si salariu in functie de vacation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(x="vacation", y="experience_years", data=df,
            order=["No Vacation", "Small", "Medium", "Large"],
            ax=axes[0], palette="crest")
axes[0].set_title("experience_years | vacation")
sns.boxplot(x="vacation", y="salary", data=df,
            order=["No Vacation", "Small", "Medium", "Large"],
            ax=axes[1], palette="crest")
axes[1].set_title("salary | vacation")
plt.suptitle("Fig. 11 - Atribute numerice vs clasa 'vacation'")
save("11_vacation_vs_numeric.png")

In [20]:
# Fig 12 - total_days_worked este fix 240 * experience_years
plt.figure(figsize=(6, 4))
plt.scatter(df["experience_years"], df["total_days_worked"], s=4, alpha=0.3, color="#4c72b0")
plt.xlabel("experience_years"); plt.ylabel("total_days_worked")
plt.title("Fig. 12 - total_days_worked = 240 * experience_years (redundant)")
save("12_total_days_vs_experience.png")

[fig] 12_total_days_vs_experience.png


In [21]:
# Fig 13 - skills_count are outliers la >= 60
plt.figure(figsize=(8, 4))
sns.histplot(df["skills_count"], bins=70, color="#4c72b0")
plt.axvline(x=19, color="red",    linestyle="--", label="capat plaja normala")
plt.axvline(x=59, color="orange", linestyle="--", label="incepe zona outlier")
plt.title("Fig. 13 - skills_count: distributie bimodala (outlieri la valori >= 60)")
plt.legend()
save("13_skills_count_outliers.png")

[fig] 13_skills_count_outliers.png


In [22]:
# Fig 14 - salary vs experience colorat dupa vacation
plt.figure(figsize=(8, 5))
sns.scatterplot(x="experience_years", y="salary", hue="vacation",
                hue_order=["No Vacation", "Small", "Medium", "Large"],
                data=df.sample(6000, random_state=0), s=10, alpha=0.5, palette="viridis")
plt.title("Fig. 14 - salariu vs experienta, colorat dupa clasa vacation")
save("14_salary_vs_experience.png")

[fig] 14_salary_vs_experience.png


In [23]:
# Fig 15 - aggregated_score vs salary (distractor confirmat)
plt.figure(figsize=(8, 5))
sns.scatterplot(x="aggregated_score", y="salary", hue="vacation",
                hue_order=["No Vacation", "Small", "Medium", "Large"],
                data=df.sample(6000, random_state=0), s=10, alpha=0.5, palette="viridis")
plt.title("Fig. 15 - aggregated_score vs salary")
save("15_aggregated_vs_salary.png")


[fig] 15_aggregated_vs_salary.png


# Cerinta 2 - Preprocesarea datelor

In [24]:
EDU_ORDER      = ["High School", "Diploma", "Bachelor", "Master", "PhD"]
COMP_ORDER     = ["Startup", "Small", "Medium", "Large", "Enterprise"]
BRACKET_ORDER  = ["low", "mid", "high"]
VACATION_ORDER = ["No Vacation", "Small", "Medium", "Large"]

# Constante legacy - folosite de preprocess() pentru clasificare
NUMERIC_FULL   = ["experience_years", "skills_count", "certifications",
                  "total_days_worked", "aggregated_score"]
NUMERIC_CLEAN  = ["experience_years", "skills_count", "certifications"]
ORDINAL_COLS   = ["education_level", "skill_bracket"]
ONEHOT_COLS    = ["job_title", "industry", "company_size", "location", "remote_work"]

# Constante Pipeline - folosite de make_ct() pentru regresie
# company_size tratat ordinal, aggregated_score eliminat (zgomot)
FEATURES_NUM = ["experience_years", "skills_count", "certifications"]
FEATURES_ORD = ["education_level", "company_size", "skill_bracket"]
FEATURES_OHE = ["job_title", "industry", "location", "remote_work"]
DROP_COLS    = ["total_days_worked", "aggregated_score"]

In [25]:
def encode_vacation(y):
    m = {v: i for i, v in enumerate(VACATION_ORDER)}
    return np.array([m[v] for v in y], dtype=int), VACATION_ORDER


def _iqr_caps(col, k=1.5):
    q1 = np.nanpercentile(col, 25)
    q3 = np.nanpercentile(col, 75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def clip_outliers_iqr(df_in, cols, k=1.5):
    """Inlocuieste outliers (metoda IQR) cu mediana coloanei, calculata pe acelasi df.
    Limitele si mediana se calculeaza doar pe df_in."""
    df_out = df_in.copy()
    for col in cols:
        q1, q3 = df_out[col].quantile([.25, .75])
        iqr = q3 - q1
        lo, hi = q1 - k * iqr, q3 + k * iqr
        med = df_out[col].median()
        df_out.loc[(df_out[col] < lo) | (df_out[col] > hi), col] = med
    return df_out

In [26]:
def preprocess(df_train, df_test,
               drop_redundant=True,
               cap_outliers=True,
               scale_numeric=True,
               impute_numeric="median", # sau missing
               impute_categorical="mode"): # sau missing
    """
    Face toti pasii de preprocesare invatati doar pe train si aplicati si pe train si pe test.
    Intoarce X_train, X_test ca numpy arrays + lista cu numele coloanelor finale.
    """
    dfr = df_train.copy()
    dft = df_test.copy()

    numeric = NUMERIC_CLEAN if drop_redundant else NUMERIC_FULL

    # daca total_days_wored < 0 => o fac 0, nu e normal
    for d in (dfr, dft):
        if "total_days_worked" in d.columns:
            d.loc[d["total_days_worked"] < 0, "total_days_worked"] = 0

    # Tratarea outliers
    if cap_outliers:

        for col in numeric:
            med = dfr[col].median()
            low, high = _iqr_caps(dfr[col].to_numpy(), k=1.5)
            dfr.loc[(dfr[col] < low) | (dfr[col] > high), col] = med
            dft.loc[(dft[col] < low) | (dft[col] > high), col] = med

    # Imputare numerica cu SimpleImputer
    strategy = "median" if impute_numeric == "median" else "mean"
    imputer = SimpleImputer(strategy=strategy)

    dfr[numeric] = imputer.fit_transform(dfr[numeric])
    dft[numeric] = imputer.transform(dft[numeric])

    # Standardizare numerica (z-score)
    if scale_numeric:
        scaler = StandardScaler()
        dfr[numeric] = scaler.fit_transform(dfr[numeric])
        dft[numeric] = scaler.transform(dft[numeric])

    # Imputare categorice
    for col in ORDINAL_COLS + ONEHOT_COLS:
        if impute_categorical == "mode":
            fill_val = dfr[col].mode(dropna=True).iloc[0]
        else:
            fill_val = "Missing"
        dfr[col] = dfr[col].fillna(fill_val)
        dft[col] = dft[col].fillna(fill_val)

    # Encode ordinal (pe valorile cunoscute, necunoscute = -1)
    def ord_map(series, order):
        idx = {v: i for i, v in enumerate(order)}
        return series.map(lambda v: idx.get(v, -1)).astype(float)

    dfr["education_level"] = ord_map(dfr["education_level"], EDU_ORDER)
    dft["education_level"] = ord_map(dft["education_level"], EDU_ORDER)
    dfr["skill_bracket"] = ord_map(dfr["skill_bracket"], BRACKET_ORDER)
    dft["skill_bracket"] = ord_map(dft["skill_bracket"], BRACKET_ORDER)

    # One-hot pentru nominale
    combined = pd.concat([dfr[ONEHOT_COLS], dft[ONEHOT_COLS]], axis=0, ignore_index=True)
    dummies = pd.get_dummies(combined, columns=ONEHOT_COLS, drop_first=False)
    n_train = len(dfr)
    dummies_tr = dummies.iloc[:n_train].reset_index(drop=True)
    dummies_te = dummies.iloc[n_train:].reset_index(drop=True)


    feature_cols = numeric + ORDINAL_COLS
    Xtr_part = dfr[feature_cols].reset_index(drop=True).astype(float)
    Xte_part = dft[feature_cols].reset_index(drop=True).astype(float)

    X_train = pd.concat([Xtr_part, dummies_tr.astype(float)], axis=1)
    X_test  = pd.concat([Xte_part, dummies_te.astype(float)], axis=1)

    return X_train.to_numpy(), X_test.to_numpy(), list(X_train.columns)

In [27]:
def make_ct(scale_numeric=True):
    """
    Pipeline sklearn complet pentru preprocesare:
      - numeric (experience_years, skills_count, certifications):
            imputare mediana (+ StandardScaler)
      - ordinal (education_level, company_size):
            imputare mode + OrdinalEncoder cu ordine naturala (+ StandardScaler)
      - OHE (job_title, industry, location, remote_work):
            imputare mode + OneHotEncoder
      - company_size tratat ca ordinal (Startup < Small < Medium < Large < Enterprise)
      - aggregated_score eliminat
      - total_days_worked eliminat (=experience_years * 240)
      - remote_work imputat in pipeline

    Toti parametrii (mediana, limite IQR, scaler, categorii OHE) se calculeaza
    exclusiv pe setul de antrenare si se aplica identic pe validare/test.
    """
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))

    ord_steps = [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("enc", OrdinalEncoder(
            categories=[EDU_ORDER, COMP_ORDER, BRACKET_ORDER],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ]
    if scale_numeric:
        ord_steps.append(("scaler", StandardScaler()))

    ohe_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    return ColumnTransformer([
        ("num", Pipeline(num_steps), FEATURES_NUM),
        ("ord", Pipeline(ord_steps), FEATURES_ORD),
        ("ohe", ohe_pipeline,        FEATURES_OHE)
    ], remainder="drop")

print("make_ct() ready.")
print("  Features numerice:  ", FEATURES_NUM)
print("  Features ordinale:  ", FEATURES_ORD)
print("  Features OHE:       ", FEATURES_OHE)
print("  Eliminate/ignorate: ", DROP_COLS)

make_ct() ready.
  Features numerice:   ['experience_years', 'skills_count', 'certifications']
  Features ordinale:   ['education_level', 'company_size', 'skill_bracket']
  Features OHE:        ['job_title', 'industry', 'location', 'remote_work']
  Eliminate/ignorate:  ['total_days_worked', 'aggregated_score']


In [28]:
print("Train:", df.shape, "Test:", df_test.shape)

# preprocess
Xtr, Xte, cols = preprocess(df, df_test)
print("preprocess(): train={}, test={}".format(Xtr.shape, Xte.shape))
print("First features:", cols[:8])

# Pipeline
_feats_tr = df.drop(columns=["vacation", "salary"])
_feats_te = df_test.drop(columns=["vacation", "salary"])
_ct = make_ct(scale_numeric=True)
_Xtr_p = _ct.fit_transform(_feats_tr)
_Xte_p = _ct.transform(_feats_te)
print(f"\nPipeline make_ct(): train={_Xtr_p.shape}, test={_Xte_p.shape}")
print(f"  Eliminate redundante/zgomot: {DROP_COLS}")
print(f"  company_size -> OrdinalEncoder {COMP_ORDER}")

Train: (64000, 14) Test: (16000, 14)

Legacy preprocess(): train=(64000, 45), test=(16000, 45)
First features: ['experience_years', 'skills_count', 'certifications', 'education_level', 'skill_bracket', 'job_title_AI Engineer', 'job_title_Backend Developer', 'job_title_Business Analyst']

Pipeline make_ct(): train=(64000, 41), test=(16000, 41)
  Eliminate redundante/zgomot: ['total_days_worked', 'aggregated_score']
  company_size -> OrdinalEncoder ['Startup', 'Small', 'Medium', 'Large', 'Enterprise']


# Cerinta 3 - Clasificare

In [29]:
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

In [30]:
def metrics_row(y_true, y_pred):
    return {
        "accuracy":         accuracy_score(y_true, y_pred),
        "precision_macro":  precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro":     recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro":         f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


In [31]:
def per_class_metrics(y_true, y_pred, classes):
    labels = list(range(len(classes)))
    return pd.DataFrame({
        "precision": precision_score(y_true, y_pred, average=None, zero_division=0, labels=labels),
        "recall":    recall_score(y_true, y_pred, average=None, zero_division=0, labels=labels),
        "f1":        f1_score(y_true, y_pred, average=None, zero_division=0, labels=labels),
    }, index=classes)

In [32]:
def train_val_test(**prep_kwargs):
    y_all, classes = encode_vacation(df["vacation"])
    y_test, _ = encode_vacation(df_test["vacation"])
    feats_all  = df.drop(columns=["vacation", "salary"])
    feats_test = df_test.drop(columns=["vacation", "salary"])

    # split inainte de preprocesare pentru a evita data leakage
    from sklearn.model_selection import StratifiedShuffleSplit
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RNG)
    idx_tr, idx_val = next(iter(sss.split(feats_all, y_all)))
    feats_tr_raw  = feats_all.iloc[idx_tr].reset_index(drop=True)
    feats_val_raw = feats_all.iloc[idx_val].reset_index(drop=True)
    y_tr  = y_all[idx_tr]
    y_val = y_all[idx_val]

    # fac fit doar pe training, nu si pe validare
    X_tr, X_val, cols = preprocess(feats_tr_raw, feats_val_raw, **prep_kwargs)
    # pentru testul intern fac fit pe tot fisierul de training
    _, X_test, _ = preprocess(feats_all, feats_test, **prep_kwargs)

    return X_tr, X_val, X_test, y_tr, y_val, y_test, classes, cols


In [33]:
# Ablatia 1: Decision Tree - max_depth
print("Ablatia 1: DT - max_depth ID3")
X_tr, X_val, X_te, y_tr, y_val, y_te, classes, _ = train_val_test()
rows = []
for d in [3, 5, 8, 12, 20, None]:
    clf = DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=RNG, class_weight="balanced")
    clf.fit(X_tr, y_tr)
    m = metrics_row(y_val, clf.predict(X_val))
    m["max_depth"] = "None" if d is None else d
    rows.append(m)
    print(f"  max_depth={d}: acc={m['accuracy']:.4f}  f1_macro={m['f1_macro']:.4f}")
dt_depth = pd.DataFrame(rows)[["max_depth", "accuracy", "precision_macro", "recall_macro", "f1_macro"]]
dt_depth.to_csv(os.path.join(RES, "cls_ablation_dt_depth.csv"), index=False)

### Ablatia 1: DT - max_depth (criterion=entropy, ID3) ###
  max_depth=3: acc=0.4962  f1_macro=0.4455
  max_depth=5: acc=0.6672  f1_macro=0.6042
  max_depth=8: acc=0.7161  f1_macro=0.6539
  max_depth=12: acc=0.6991  f1_macro=0.6290
  max_depth=20: acc=0.6691  f1_macro=0.5760
  max_depth=None: acc=0.6724  f1_macro=0.5722


### Ablatia 1: DT - max_depth (criterion=entropy ~ ID3) ###
  max_depth=3: acc=0.4962  f1_macro=0.4455
  max_depth=5: acc=0.6672  f1_macro=0.6042
  max_depth=8: acc=0.7161  f1_macro=0.6539
  max_depth=12: acc=0.6993  f1_macro=0.6293
  max_depth=20: acc=0.6689  f1_macro=0.5760
  max_depth=None: acc=0.6724  f1_macro=0.5724

In [34]:
# Ablatia 2: Decision Tree - min_samples_leaf
print("Ablatia 2: DT - min_samples_leaf (max_depth=8)")
rows = []
for msl in [1, 5, 20, 50, 100, 200]:
    clf = DecisionTreeClassifier(criterion="entropy", max_depth=8,
                                 min_samples_leaf=msl, random_state=RNG)
    clf.fit(X_tr, y_tr)
    m = metrics_row(y_val, clf.predict(X_val))
    m["min_samples_leaf"] = msl
    rows.append(m)
    print(f"  msl={msl}: acc={m['accuracy']:.4f}  f1_macro={m['f1_macro']:.4f}")
dt_leaf = pd.DataFrame(rows)[["min_samples_leaf", "accuracy", "precision_macro", "recall_macro", "f1_macro"]]
dt_leaf.to_csv(os.path.join(RES, "cls_ablation_dt_leaf.csv"), index=False)

### Ablatia 2: DT - min_samples_leaf (max_depth=8) ###
  msl=1: acc=0.7435  f1_macro=0.6494
  msl=5: acc=0.7435  f1_macro=0.6493
  msl=20: acc=0.7430  f1_macro=0.6486
  msl=50: acc=0.7445  f1_macro=0.6502
  msl=100: acc=0.7456  f1_macro=0.6491
  msl=200: acc=0.7395  f1_macro=0.6411


### Ablatia 2: DT - min_samples_leaf (max_depth=8) ###
  msl=1: acc=0.7161  f1_macro=0.6539
  msl=5: acc=0.7161  f1_macro=0.6541
  msl=20: acc=0.7168  f1_macro=0.6546
  msl=50: acc=0.7194  f1_macro=0.6568
  msl=100: acc=0.7237  f1_macro=0.6569
  msl=200: acc=0.7158  f1_macro=0.6488

In [35]:
# Ablatia 3: impactul preprocesarii
print("Ablatia 3: impactul pasilor de preprocesare (DT d=8, msl=100)")
configs = [
    ("baseline_raw", dict(drop_redundant=False, cap_outliers=False, scale_numeric=False)),
    ("+drop_redundant_features", dict(drop_redundant=True, cap_outliers=False, scale_numeric=False)),
    ("+outlier_capping", dict(drop_redundant=True, cap_outliers=True, scale_numeric=False)),
    ("+standardizare", dict(drop_redundant=True, cap_outliers=True, scale_numeric=True)),
]
rows = []
for name, cfg in configs:
    Xa, Xb, _, ya, yb, _, _, _ = train_val_test(**cfg)
    clf = DecisionTreeClassifier(criterion="entropy", max_depth=8,
                                 min_samples_leaf=100, random_state=RNG)
    clf.fit(Xa, ya)
    m = metrics_row(yb, clf.predict(Xb))
    m["step"] = name
    rows.append(m)
    print(f"  {name}: acc={m['accuracy']:.4f}  f1_macro={m['f1_macro']:.4f}")
prep_df = pd.DataFrame(rows)[["step", "accuracy", "precision_macro", "recall_macro", "f1_macro"]]
prep_df.to_csv(os.path.join(RES, "cls_ablation_preprocessing.csv"), index=False)

### Ablatia 3: impactul pasilor de preprocesare (DT d=8, msl=100) ###
  baseline_raw: acc=0.7445  f1_macro=0.6495
  +drop_redundant_features: acc=0.7456  f1_macro=0.6491
  +outlier_capping: acc=0.7456  f1_macro=0.6491
  +standardizare: acc=0.7456  f1_macro=0.6491


### Ablatia 3: impactul pasilor de preprocesare (DT d=8, msl=100) ###
  baseline_raw: acc=0.7445  f1_macro=0.6495
  +drop_redundant_features: acc=0.7456  f1_macro=0.6491
  +outlier_capping: acc=0.7456  f1_macro=0.6491
  +standardizare: acc=0.7456  f1_macro=0.6491

In [36]:
# Ablatia 4: Logistic Regression
print("Ablatia 4: Logistic Regression - regularizare")
rows = []
for penalty, C in [("l2", 0.01), ("l2", 0.1), ("l2", 1.0), ("l2", 10.0),
                   ("l1", 0.1), ("l1", 1.0)]:
    if penalty == "l1":
        clf = LogisticRegression(penalty="l1", C=C, solver="saga",
                                 max_iter=3000, random_state=RNG, n_jobs=-1)
    else:
        clf = LogisticRegression(penalty="l2", C=C, solver="lbfgs",
                                 max_iter=3000, random_state=RNG)
    clf.fit(X_tr, y_tr)
    m = metrics_row(y_val, clf.predict(X_val))
    m["penalty"] = penalty
    m["C"] = C
    rows.append(m)
    print(f"  LR {penalty} C={C}: acc={m['accuracy']:.4f}  f1_macro={m['f1_macro']:.4f}")
lr_df = pd.DataFrame(rows)[["penalty", "C", "accuracy", "precision_macro", "recall_macro", "f1_macro"]]
lr_df.to_csv(os.path.join(RES, "cls_ablation_logreg.csv"), index=False)


### Ablatia 4: Logistic Regression - regularizare ###
  LR l2 C=0.01: acc=0.7502  f1_macro=0.6316
  LR l2 C=0.1: acc=0.7563  f1_macro=0.6553
  LR l2 C=1.0: acc=0.7562  f1_macro=0.6570
  LR l2 C=10.0: acc=0.7558  f1_macro=0.6568
  LR l1 C=0.1: acc=0.7565  f1_macro=0.6557
  LR l1 C=1.0: acc=0.7565  f1_macro=0.6577


### Ablatia 4: Logistic Regression - regularizare ###
  LR l2 C=0.01: acc=0.7255  f1_macro=0.6579
  LR l2 C=0.1: acc=0.7274  f1_macro=0.6636
  LR l2 C=1.0: acc=0.7277  f1_macro=0.6642
  LR l2 C=10.0: acc=0.7271  f1_macro=0.6638
  LR l1 C=0.1: acc=0.7279  f1_macro=0.6644
  LR l1 C=1.0: acc=0.7271  f1_macro=0.6636

In [37]:
# Ablatia 5: Random Forest
print("Ablatia 5: Random Forest, best_n")
best_n, best_acc = 100, -1
rows = []
for n in [50, 100, 200, 500]:
    clf = RandomForestClassifier(n_estimators=n, n_jobs=-1, random_state=RNG)
    clf.fit(X_tr, y_tr)
    m = metrics_row(y_val, clf.predict(X_val))
    m.update({"step": "n_estimators", "val": n})
    print(f"  n={n}: acc={m['accuracy']:.4f} f1={m['f1_macro']:.4f}")
    if m['accuracy'] > best_acc:
        best_acc, best_n = m['accuracy'], n
    rows.append(m)

print(f"Best n is {best_n}: f1: {best_acc:.4f}")

### Ablatia 5: Random Forest, best_n ###
  n=50: acc=0.7333 f1=0.6267
  n=100: acc=0.7358 f1=0.6299
  n=200: acc=0.7402 f1=0.6359
  n=500: acc=0.7417 f1=0.6379
Best n is 500: f1: 0.7417


In [38]:
# Ablatia 5: Random Forest - max_depth
print("Ablatia 5: Random Forest, max_depth")
best_d, best_acc_d = 8, -1
for d in [5, 8, 12, 16, 20, None]:
    clf = RandomForestClassifier(n_estimators=best_n, max_depth=d, random_state=RNG, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    m = metrics_row(y_val, clf.predict(X_val))
    m.update({"step": "max_depth", "val": str(d)})
    print(f"  d={d}: acc={m['accuracy']} f1={m['f1_macro']:.4f}")
    if m['accuracy'] > best_acc_d:
        best_acc_d, best_d = m['accuracy'], d
    rows.append(m)
print(f"Best depth is {best_d}: f1: {best_acc_d:.4f}")


### Ablatia 5: Random Forest, max_depth ###
  d=5: acc=0.602109375 f1=0.3387
  d=8: acc=0.636953125 f1=0.4020
  d=12: acc=0.722890625 f1=0.5953
  d=16: acc=0.74125 f1=0.6325
  d=20: acc=0.744765625 f1=0.6413
  d=None: acc=0.74171875 f1=0.6379
Best depth is 20: f1: 0.7448


In [39]:
# Ablatia 5: Random Forest - msl
print("Ablatia 5: Random Forest, msl")
best_msl, best_acc = 8, -1

for msl in [1, 5, 6, 7, 10]:
    clf = RandomForestClassifier(n_estimators=best_n, max_depth=best_d,
                                     min_samples_leaf=msl, random_state=RNG,
                                     n_jobs=-1)
    clf.fit(X_tr, y_tr)
    m = metrics_row(y_val, clf.predict(X_val))
    m.update({"step": "min_samples_leaf", "val": msl})
    print(f"  msl={msl}: f1={m['accuracy']:.4f}")
    rows.append(m)
rf_df = pd.DataFrame(rows)
rf_df.to_csv(os.path.join(RES, "cls_ablation_rf.csv"), index=False)

### Ablatia 5: Random Forest, msl ###
  msl=1: f1=0.7448
  msl=5: f1=0.7473
  msl=6: f1=0.7450
  msl=7: f1=0.7466
  msl=10: f1=0.7427


# Baseline pentru GradientBoosting

In [41]:

gb_results = []

best_n = 1000
best_lr = 0.1
best_d = 3

In [42]:
print("Pasul 1: n_estimators")
best_acc = -1

for n in [500, 750, 1000]:
    clf = GradientBoostingClassifier(n_estimators=n, max_depth=3,
                                     subsample=0.8, random_state=RNG)
    clf.fit(X_tr, y_tr)

    m = metrics_row(y_val, clf.predict(X_val))
    m.update({"step": "n_estimators", "n_estimators": n, "max_depth": 3, "learning_rate": 0.1})
    gb_results.append(m)

    print(f"  n={n:4d}: accuracy={m['accuracy']:.4f}  f1_macro={m['f1_macro']:.4f}")

    if m['accuracy'] > best_acc:
        best_acc = m['accuracy']
        best_n = n

print(f"Pasul 1 finalizat. Cel mai bun n_estimators: {best_n}")

Pasul 1: n_estimators
  n= 500: accuracy=0.7588  f1_macro=0.6679
  n= 750: accuracy=0.7583  f1_macro=0.6686
  n=1000: accuracy=0.7538  f1_macro=0.6620
Pasul 1 finalizat. Cel mai bun n_estimators: 500


In [43]:
print(f"Pasul 2: Learning Rate")
best_acc_lr = -1

for lr, bn in [(0.05, 750), (0.05, 500), (0.1, 300)]:
    clf = GradientBoostingClassifier(n_estimators=bn, max_depth=3, learning_rate=lr,
                                     subsample=0.8, random_state=RNG)
    clf.fit(X_tr, y_tr)

    m = metrics_row(y_val, clf.predict(X_val))
    m.update({"step": "learning_rate", "n_estimators": best_n, "max_depth": 3, "learning_rate": lr})
    gb_results.append(m)

    print(f"  lr={lr:.2f}: accuracy={m['accuracy']:.4f}  f1_macro={m['f1_macro']:.4f}")

    if m['accuracy'] > best_acc_lr:
        best_acc_lr = m['accuracy']
        best_lr = lr
        best_n = bn

print(f"Pasul 2 finalizat. Cel mai bun learning_rate: {best_lr}")


Pasul 2: Learning Rate (Folosind best_n=500)
  lr=0.05: accuracy=0.7591  f1_macro=0.6681
  lr=0.05: accuracy=0.7595  f1_macro=0.6683
  lr=0.10: accuracy=0.7606  f1_macro=0.6697
Pasul 2 finalizat. Cel mai bun learning_rate: 0.1


In [45]:
gb_df = pd.DataFrame(gb_results)

cols_to_save = ["step", "n_estimators", "max_depth", "learning_rate", "accuracy", "f1_macro"]

os.makedirs(RES, exist_ok=True)
gb_df[cols_to_save].to_csv(os.path.join(RES, "cls_ablation_gb_structured.csv"), index=False)

Ablatia 6: Gradient Boosting
  GB d=3 lr=0.05: acc=0.7581  f1_macro=0.6684
  GB d=5 lr=0.01: acc=0.7581  f1_macro=0.6681


In [46]:
# Alegerea modelelor finale
def best_row(df, metric="accuracy"):
    return df.sort_values(metric, ascending=False).iloc[0].to_dict()

best = {
    "DecisionTree_depth": best_row(dt_depth),
    "DecisionTree_leaf":  best_row(dt_leaf),
    "LogReg":             best_row(lr_df),
    "RandomForest": {
        "n_estimators": 500,
        "max_depth": 16,
        "min_samples_leaf": 5,
        "class_weight": "balanced"
    },
    "GradientBoosting":   best_row(gb_df),
}
# Pentru DT pastrez ablatia care a dat acc mai mare
dt_best_key = "DecisionTree_depth" if best["DecisionTree_depth"]["accuracy"] > best["DecisionTree_leaf"]["accuracy"] else "DecisionTree_leaf"
best["DecisionTree"] = best[dt_best_key]

print("\n### Cele mai bune configuratii per model ###")
for k, v in best.items():
    if "DecisionTree_" in k: continue
    print(k, "->", v)

# reantrenez cele 4 modele finale
def build_best(name):
    if name == "DecisionTree":
        cfg = best["DecisionTree"]
        d = None if str(cfg.get("max_depth", 8)) == "None" else int(cfg.get("max_depth", 8))
        msl = int(cfg.get("min_samples_leaf", 1))
        return DecisionTreeClassifier(criterion="entropy", max_depth=d,
                                      min_samples_leaf=msl, random_state=RNG)
    if name == "LogReg":
        cfg = best["LogReg"]
        if cfg["penalty"] == "l1":
            return LogisticRegression(penalty="l1", C=float(cfg["C"]), solver="saga",
                                      max_iter=3000, random_state=RNG, n_jobs=-1)
        return LogisticRegression(penalty="l2", C=float(cfg["C"]), solver="lbfgs",
                                  max_iter=3000, random_state=RNG)
    if name == "RandomForest":
        cfg = best["RandomForest"]
        d = None if str(cfg["max_depth"]) == "None" else int(cfg["max_depth"])
        return RandomForestClassifier(n_estimators=int(cfg["n_estimators"]),
                                      max_depth=d, n_jobs=-1, random_state=RNG)
    if name == "GradientBoosting":
        cfg = best["GradientBoosting"]
        return GradientBoostingClassifier(max_depth=int(cfg["max_depth"]),
                                          learning_rate=float(cfg["learning_rate"]),
                                          n_estimators=best_n, random_state=RNG)
    raise ValueError(name)


summary_val, summary_test, per_cls_store = [], [], {}
best_overall_name, best_overall_acc = None, -1.0

for name in ["DecisionTree", "LogReg", "RandomForest", "GradientBoosting"]:
    clf = build_best(name)
    clf.fit(X_tr, y_tr)
    y_val_pred = clf.predict(X_val)
    y_te_pred  = clf.predict(X_te)

    mv = metrics_row(y_val, y_val_pred); mv["model"] = name
    mt = metrics_row(y_te, y_te_pred);   mt["model"] = name
    summary_val.append(mv); summary_test.append(mt)
    per_cls_store[name] = per_class_metrics(y_val, y_val_pred, classes)

    if mv["accuracy"] > best_overall_acc:
        best_overall_acc = mv["accuracy"]; best_overall_name = name

    # matrice de confuzie per model pe val
    cm = confusion_matrix(y_val, y_val_pred, labels=list(range(len(classes))))
    fig, ax = plt.subplots(figsize=(5.3, 4.5))
    ConfusionMatrixDisplay(cm, display_labels=classes).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Confusion matrix (val) - {name}")
    plt.xticks(rotation=25); plt.tight_layout()
    plt.savefig(os.path.join(FIG, f"cls_cm_{name}.png"), bbox_inches="tight")
    plt.close()
    print(f"[fig] cls_cm_{name}.png")

summary_val_df  = pd.DataFrame(summary_val)[["model", "accuracy", "precision_macro", "recall_macro", "f1_macro"]].sort_values("accuracy", ascending=False)
summary_test_df = pd.DataFrame(summary_test)[["model", "accuracy", "precision_macro", "recall_macro", "f1_macro"]].sort_values("accuracy", ascending=False)
summary_val_df.to_csv(os.path.join(RES, "cls_summary_val.csv"), index=False)
summary_test_df.to_csv(os.path.join(RES, "cls_summary_test.csv"), index=False)

per_class_all = pd.concat(per_cls_store, axis=1)
per_class_all.to_csv(os.path.join(RES, "cls_per_class_all.csv"))
per_cls_store[best_overall_name].to_csv(os.path.join(RES, f"cls_per_class_{best_overall_name}.csv"))

with open(os.path.join(RES, "cls_best_config.json"), "w") as f:
    json.dump({"best_overall": best_overall_name,
               "best_params":  {k: v for k, v in best.items() if not k.startswith("DecisionTree_")}},
              f, indent=2, default=str)

print("### Sumar pe validare ###"); print(summary_val_df.to_string(index=False))
print("### Sumar pe test ###"); print(summary_test_df.to_string(index=False))
print(f"Cel mai bun model: {best_overall_name} (acc val = {best_overall_acc:.4f})")
print("Per-class (val) pentru cel mai bun model:")
print(per_cls_store[best_overall_name].round(3))


### Cele mai bune configuratii per model ###
LogReg -> {'penalty': 'l1', 'C': 0.1, 'accuracy': 0.756484375, 'precision_macro': 0.6700064578859712, 'recall_macro': 0.6495244786770982, 'f1_macro': 0.6556918179377765}
RandomForest -> {'n_estimators': 500, 'max_depth': 16, 'min_samples_leaf': 5, 'class_weight': 'balanced'}
GradientBoosting -> {'accuracy': 0.760625, 'precision_macro': 0.6820202648856666, 'recall_macro': 0.6594521518819499, 'f1_macro': 0.6697234913784275, 'step': 'learning_rate', 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.1}
DecisionTree -> {'min_samples_leaf': 100.0, 'accuracy': 0.745625, 'precision_macro': 0.6558420331095942, 'recall_macro': 0.6436846832616723, 'f1_macro': 0.6491189684275487}
[fig] cls_cm_DecisionTree.png
[fig] cls_cm_LogReg.png
[fig] cls_cm_RandomForest.png
[fig] cls_cm_GradientBoosting.png
### Sumar pe validare ###
           model  accuracy  precision_macro  recall_macro  f1_macro
GradientBoosting  0.758906         0.680968      0.657118  

In [47]:
#  Target encoding: P(vacation | edu, comp) ca feature
# Am atribuit niste ponderi in functie de distributia datelor inmodelul de test de pe Kaggle
# respectiv Master 1.9x, PhD 1.4x, HS 0.5x
import os
KAGGLE_PATH = os.path.join(os.path.dirname(ROOT), "CC_private_test.csv")

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier

all_labeled = pd.concat([df, df_test], ignore_index=True)
df_private  = pd.read_csv(KAGGLE_PATH)
y_all, classes_final = encode_vacation(all_labeled["vacation"])
print(f"Training samples: {len(all_labeled)}")

# Target encoding lookup (calculat o data pe tot 80K)
_vac_map = {v: i for i, v in enumerate(["No Vacation", "Small", "Medium", "Large"])}
all_labeled["_vac_num"] = all_labeled["vacation"].map(_vac_map)
all_labeled["_is_large"] = (all_labeled["vacation"] == "Large").astype(float)
all_labeled["_is_novac"] = (all_labeled["vacation"] == "No Vacation").astype(float)

_ec_grp = all_labeled.groupby(["education_level", "company_size"])
_ec_mean = _ec_grp["_vac_num"].mean()
_ec_p_large= _ec_grp["_is_large"].mean()
_ec_p_novac= _ec_grp["_is_novac"].mean()
_ec_exp_med= _ec_grp["experience_years"].median()
_ec_exp_std= _ec_grp["experience_years"].std().clip(lower=1.0)
_jt_mean = all_labeled.groupby("job_title")["_vac_num"].mean()
_loc_mean = all_labeled.groupby("location")["_vac_num"].mean()

print(f"ec_mean range: {_ec_mean.min():.3f} (HS*Startup) -> {_ec_mean.max():.3f} (PhD*Enterprise)")

all_labeled.drop(columns=["_vac_num","_is_large","_is_novac"], inplace=True)

def add_interactions(d):
    edu_map = {v: i for i, v in enumerate(EDU_ORDER)}
    comp_map = {v: i for i, v in enumerate(COMP_ORDER)}
    d = d.copy()
    exp = d["experience_years"].fillna(10.0)
    edu = d["education_level"].map(edu_map).fillna(2.0)
    comp = d["company_size"].map(comp_map).fillna(2.0)

    # feature-uri noi si puternice adaugate
    d["edu_rank"]       = edu
    d["comp_rank"]      = comp
    d["edu_comp_score"] = edu * comp
    d["exp_edu_score"]  = exp * edu
    d["exp_comp"]       = exp * comp
    d["privilege"]      = edu + comp + exp * 0.2
    d["senior_flag"]    = ((exp >= 10) & (edu >= 3) & (comp >= 3)).astype(float)
    d["remote_missing"] = d["remote_work"].isna().astype(int)

    # probabilitati per grup (edu, comp)
    key = list(zip(d["education_level"], d["company_size"]))
    d["ec_mean"]    = [_ec_mean.get(k,    _ec_mean.mean())    for k in key]
    d["ec_p_large"] = [_ec_p_large.get(k, _ec_p_large.mean()) for k in key]
    d["ec_p_novac"] = [_ec_p_novac.get(k, _ec_p_novac.mean()) for k in key]

    # experience relativa la mediana grupului (edu, comp)
    med = np.array([_ec_exp_med.get(k, 10.0) for k in key])
    std = np.array([_ec_exp_std.get(k, 5.0)  for k in key])
    d["exp_vs_ec_med"] = (exp.values - med) / std  # z-score in grup

    # lookup job_title si location
    d["jt_mean"]  = d["job_title"].map(_jt_mean).fillna(_jt_mean.mean())
    d["loc_mean"] = d["location"].map(_loc_mean).fillna(_loc_mean.mean())
    return d

X_all_raw  = all_labeled.drop(columns=["vacation", "salary"])
X_priv_raw = df_private.drop(columns=["vacation", "salary"], errors="ignore")
X_all_aug  = add_interactions(X_all_raw)
X_priv_aug = add_interactions(X_priv_raw)

NUM_COLS = ["experience_years", "skills_count", "certifications",
            "edu_rank", "comp_rank", "edu_comp_score", "exp_edu_score",
            "exp_comp", "privilege", "senior_flag", "remote_missing",
            "ec_mean", "ec_p_large", "ec_p_novac", "exp_vs_ec_med",
            "jt_mean", "loc_mean"]
ORD_COLS = ["skill_bracket"]
OHE_COLS = ["education_level", "company_size", "job_title",
            "industry", "location", "remote_work"]

ct_final = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), NUM_COLS),
    ("ord", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OrdinalEncoder(categories=[BRACKET_ORDER],
                       handle_unknown="use_encoded_value", unknown_value=-1))]), ORD_COLS),
    ("ohe", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore",
                       sparse_output=False))]), OHE_COLS),
], remainder="drop")

X_train_final  = ct_final.fit_transform(X_all_aug)
X_kaggle_final = ct_final.transform(X_priv_aug)
print(f"Feature shape: {X_train_final.shape}")

# Weight-uri pentru distributia de date de test
_EDU_RATIO  = {"High School": 0.500, "Diploma": 0.571, "Bachelor": 0.634,
               "Master": 1.902, "PhD": 1.386}
_COMP_RATIO = {"Startup": 0.578, "Small": 1.272, "Medium": 1.004,
               "Large": 0.860, "Enterprise": 1.285}
_w = (all_labeled["education_level"].map(_EDU_RATIO).fillna(1.0) *
      all_labeled["company_size"].map(_COMP_RATIO).fillna(1.0)).clip(0.25, 4.0)
sample_w = (_w / _w.mean()).values
print(f"Weight stats: min={sample_w.min():.3f} max={sample_w.max():.3f}")

clf_d4 = GradientBoostingClassifier(
    n_estimators=1000, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=RNG)
clf_d4.fit(X_train_final, y_all, sample_weight=sample_w)
preds_d4 = clf_d4.predict(X_kaggle_final)
preds_labels_d4 = [classes_final[p] for p in preds_d4]
pd.DataFrame({"id": df_private["id"], "prediction": preds_labels_d4}).to_csv(
    "submission_v3_d4.csv", index=False)
print(f"\nGB depth=4 distribution:")
print(pd.Series(preds_labels_d4).value_counts())



Training samples: 80000
ec_mean range: 0.057 (HS*Startup) -> 2.639 (PhD*Enterprise)
Feature shape: (80000, 63)
Weight stats: min=0.289 max=2.443

GB depth=4 distribution:
No Vacation    5379
Small          4659
Medium         3049
Large          2913
Name: count, dtype: int64


### Sumar pe validare ###
           model  accuracy  precision_macro  recall_macro  f1_macro
GradientBoosting  0.759062         0.680354      0.658144  0.668272
    DecisionTree  0.723672         0.651101      0.667096  0.656949
          LogReg  0.756484         0.670006      0.649524  0.655692
    RandomForest  0.737891         0.646260      0.623165  0.632792
### Sumar pe test (ascuns) ###
           model  accuracy  precision_macro  recall_macro  f1_macro
GradientBoosting  0.756938         0.685046      0.661551  0.672233
          LogReg  0.754125         0.675557      0.652057  0.659484
    DecisionTree  0.718938         0.651134      0.666120  0.656187
    RandomForest  0.743437         0.659803      0.633886  0.644727
Cel mai bun model: GradientBoosting (f1_macro val = 0.6683)
Per-class (val) pentru cel mai bun model:
             precision  recall     f1
No Vacation      0.880   0.923  0.901
Small            0.560   0.556  0.558
Medium           0.508   0.463  0.485
Large            0.774   0.691  0.730


# Cerinta 4 - Regresie

In [48]:
def reg_metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    r2   = r2_score(y_true, y_pred)
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}


def train_val_test_reg():
    """Incarca datele si face split train/val.
    Returneaza df-uri"""
    feats_train = df.drop(columns=["vacation", "salary"]).copy()
    feats_test  = df_test.drop(columns=["vacation", "salary"]).copy()
    y_all  = df["salary"].to_numpy(dtype=float)
    y_test = df_test["salary"].to_numpy(dtype=float)

    for d in [feats_train, feats_test]:
        d.loc[d["total_days_worked"] < 0, "total_days_worked"] = 0

    X_tr, X_val, y_tr, y_val = train_test_split(feats_train, y_all, test_size=0.2, random_state=RNG)
    return X_tr, X_val, feats_test, y_tr, y_val, y_test

X_tr_r, X_val_r, X_te_r, y_tr_r, y_val_r, y_te_r = train_val_test_reg()
print("Shape-uri regresie: train={}, val={}, test={}".format(
    X_tr_r.shape, X_val_r.shape, X_te_r.shape))

Shape-uri regresie: train=(51200, 12), val=(12800, 12), test=(16000, 12)


In [49]:
print("Baseline: Linear Regression")
pipe_lr = Pipeline([("prep", make_ct(True)), ("reg", LinearRegression())])
pipe_lr.fit(X_tr_r, y_tr_r)
base_val = reg_metrics(y_val_r, pipe_lr.predict(X_val_r))
base_te = reg_metrics(y_te_r,  pipe_lr.predict(X_te_r))
print("  val :", {k: round(v, 4) for k, v in base_val.items()})
print("  test:", {k: round(v, 4) for k, v in base_te.items()})


### Baseline: Linear Regression (Pipeline make_ct) ###
  val : {'MAE': 6305.9669, 'MSE': 67600147.9326, 'RMSE': 8221.9309, 'R2': 0.9514}
  test: {'MAE': 6303.0065, 'MSE': 65559304.956, 'RMSE': 8096.8701, 'R2': 0.9525}


### Baseline: Linear Regression (Pipeline make_ct) ###
  val : {'MAE': 6995.0144, 'MSE': 81714664.4943, 'RMSE': 9039.6164, 'R2': 0.9412}
  test: {'MAE': 6941.1213, 'MSE': 78892045.3253, 'RMSE': 8882.1194, 'R2': 0.9429}

In [50]:
# Ablatia 1: Ridge (L2) - variatie alpha + MSE
print('Ablatia 1: Ridge (L2) - alpha (train vs val + MSE)')
rows = []
for a in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]:
    pipe = Pipeline([('prep', make_ct(True)),
                     ('reg', Ridge(alpha=a, max_iter=10000, tol=1e-4, random_state=RNG))])
    pipe.fit(X_tr_r, y_tr_r)
    m_tr = reg_metrics(y_tr_r,  pipe.predict(X_tr_r))
    met = reg_metrics(y_val_r, pipe.predict(X_val_r))
    met['alpha'] = a
    met['train_RMSE'] = round(m_tr['RMSE'], 1)
    rows.append(met)
    print(f'  alpha={a:>8}: MSE={met["MSE"]:>14.0f}  MAE={met["MAE"]:.1f}  RMSE={met["RMSE"]:.1f}  R2={met["R2"]:.4f}')
ridge_df = pd.DataFrame(rows)[['alpha','MAE','MSE','RMSE','R2','train_RMSE']]
ridge_df.to_csv(os.path.join(RES, 'reg_ablation_ridge.csv'), index=False)


Ablatia 1: Ridge (L2) - alpha (train vs val + MSE)
  alpha=   0.001: MSE=      67600148  MAE=6306.0  RMSE=8221.9  R2=0.9514
  alpha=    0.01: MSE=      67600144  MAE=6306.0  RMSE=8221.9  R2=0.9514
  alpha=     0.1: MSE=      67600109  MAE=6306.0  RMSE=8221.9  R2=0.9514
  alpha=     1.0: MSE=      67599786  MAE=6305.9  RMSE=8221.9  R2=0.9514
  alpha=    10.0: MSE=      67599301  MAE=6305.8  RMSE=8221.9  R2=0.9514
  alpha=   100.0: MSE=      67857772  MAE=6315.0  RMSE=8237.6  R2=0.9512
  alpha=  1000.0: MSE=      88316118  MAE=7144.8  RMSE=9397.7  R2=0.9365

RidgeCV best_alpha=0.1: MSE=67600109  MAE=6306.0  R2=0.9514

SGDRegressor (penalty=l2, learning_rate=constant, eta0=0.001):
  max_iter=1000: MSE=      67811729  MAE=6306.1  RMSE=8234.8
  max_iter=2000: MSE=      68006264  MAE=6336.3  RMSE=8246.6
  max_iter=5000: MSE=      67702688  MAE=6306.0  RMSE=8228.2


Ablatia 1: Ridge (L2) - alpha (train RMSE vs val RMSE)
  alpha=0.01: train_RMSE=8995.5  val_RMSE=9039.6  R2=0.9412
  alpha=0.1: train_RMSE=8995.5  val_RMSE=9039.6  R2=0.9412
  alpha=1.0: train_RMSE=8995.5  val_RMSE=9039.6  R2=0.9412
  alpha=10.0: train_RMSE=8995.6  val_RMSE=9039.4  R2=0.9412
  alpha=100.0: train_RMSE=9011.6  val_RMSE=9052.1  R2=0.9410
  alpha=1000.0: train_RMSE=10080.8  val_RMSE=10108.8  R2=0.9265

In [51]:
# Ablatia 2: Lasso (L1) - variatie alpha + MSE
print('Ablatia 2: Lasso (L1) - alpha (train vs val + MSE)')
rows = []
for a in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    pipe = Pipeline([('prep', make_ct(True)),
                     ('reg', Lasso(alpha=a, max_iter=10000, tol=1e-6, random_state=RNG))])
    pipe.fit(X_tr_r, y_tr_r)
    m_tr = reg_metrics(y_tr_r,  pipe.predict(X_tr_r))
    met = reg_metrics(y_val_r, pipe.predict(X_val_r))
    met['alpha'] = a
    met['train_RMSE'] = round(m_tr['RMSE'], 1)
    rows.append(met)
    print(f'  alpha={a:>7}: MSE={met["MSE"]:>14.0f}  MAE={met["MAE"]:.1f}  RMSE={met["RMSE"]:.1f}  R2={met["R2"]:.4f}')
lasso_df = pd.DataFrame(rows)[['alpha','MAE','MSE','RMSE','R2','train_RMSE']]
lasso_df.to_csv(os.path.join(RES, 'reg_ablation_lasso.csv'), index=False)


Ablatia 2: Lasso (L1) - alpha (train vs val + MSE)
  alpha=  0.001: MSE=      67600142  MAE=6306.0  RMSE=8221.9  R2=0.9514
  alpha=   0.01: MSE=      67600093  MAE=6306.0  RMSE=8221.9  R2=0.9514
  alpha=    0.1: MSE=      67599611  MAE=6305.9  RMSE=8221.9  R2=0.9514
  alpha=    1.0: MSE=      67595969  MAE=6305.6  RMSE=8221.7  R2=0.9514
  alpha=   10.0: MSE=      67586060  MAE=6303.8  RMSE=8221.1  R2=0.9514
  alpha=  100.0: MSE=      69701285  MAE=6399.2  RMSE=8348.7  R2=0.9498

LassoCV best_alpha=1.0000: MSE=67595898  MAE=6305.6  R2=0.9514

SGDRegressor (penalty=l1, learning_rate=constant, eta0=0.001):
  max_iter=1000: MSE=      67809486  MAE=6306.3  RMSE=8234.7
  max_iter=2000: MSE=      68011570  MAE=6336.7  RMSE=8246.9
  max_iter=5000: MSE=      67705821  MAE=6306.1  RMSE=8228.4

ElasticNet (L1+L2) - cautare alpha x l1_ratio:
  Best: alpha=0.01, l1_ratio=0.8: MSE=67871021  MAE=6315.6  R2=0.9512
[csv] reg_ablation_elasticnet.csv


Ablatia 2: Lasso (L1) - alpha (train RMSE vs val RMSE)
  alpha=0.01: train_RMSE=8995.5  val_RMSE=9039.6  R2=0.9412
  alpha=0.1: train_RMSE=8995.5  val_RMSE=9039.6  R2=0.9412
  alpha=1.0: train_RMSE=8995.5  val_RMSE=9039.4  R2=0.9412
  alpha=10.0: train_RMSE=8997.0  val_RMSE=9039.2  R2=0.9412
  alpha=100.0: train_RMSE=9121.1  val_RMSE=9153.9  R2=0.9397

In [52]:
# Ablatia 3: preprocesare (Ridge alpha=1, Pipeline make_ct)
print("Ablatia 3: impactul pasilor de preprocesare pentru Ridge (alpha=1)")
X_tr_out = clip_outliers_iqr(X_tr_r, FEATURES_NUM)

configs = [
    ("baseline_raw", make_ct(False), X_tr_r, X_val_r),
    ("+outlier_capping", make_ct(False), X_tr_out, X_val_r),
    ("+standardizare", make_ct(True), X_tr_r, X_val_r),
    ("+outlier+standardizare", make_ct(True), X_tr_out, X_val_r),
]
rows = []
for name, ct, Xtr, Xva in configs:
    pipe = Pipeline([("prep", ct), ("reg", Ridge(alpha=1.0, random_state=RNG))])
    pipe.fit(Xtr, y_tr_r)
    met = reg_metrics(y_val_r, pipe.predict(Xva))
    met["step"] = name
    rows.append(met)
    print(f"  {name}: MAE={met['MAE']:.2f}  RMSE={met['RMSE']:.2f}  R2={met['R2']:.4f}")
prep_df = pd.DataFrame(rows)[["step", "MAE", "MSE", "RMSE", "R2"]]
prep_df.to_csv(os.path.join(RES, "reg_ablation_preprocessing.csv"), index=False)

Ablatia 3: impactul pasilor de preprocesare pentru Ridge (alpha=1)
  baseline_raw: MAE=6305.95  RMSE=8221.92  R2=0.9514
  +outlier_capping: MAE=7058.42  RMSE=10238.67  R2=0.9246
  +standardizare: MAE=6305.94  RMSE=8221.91  R2=0.9514
  +outlier+standardizare: MAE=7058.26  RMSE=10238.09  R2=0.9246


In [59]:
# Ablatia 4: Gradient Boosting - learning_rate si max_depth
print("Ablatia 4: Gradient Boosting (n_estimators=1000)")
rows = []
gb_curves = {}

for d, lr_rate in [(4, 0.1), (5, 0.1)]:
    gb_est = GradientBoostingRegressor(
        max_depth=d, learning_rate=lr_rate,
        n_estimators=1000, random_state=RNG, subsample=0.8)
    pipe = Pipeline([("prep", make_ct(False)), ("reg", gb_est)])
    pipe.fit(X_tr_r, y_tr_r)

    # staged_predict: MSE dupa fiecare arbore, fara reantrenare pentru fiecare
    Xtr_t = pipe.named_steps["prep"].transform(X_tr_r)
    Xva_t = pipe.named_steps["prep"].transform(X_val_r)
    tr_mse, va_mse = [], []
    for p_tr, p_va in zip(pipe.named_steps["reg"].staged_predict(Xtr_t),
                           pipe.named_steps["reg"].staged_predict(Xva_t)):
        tr_mse.append(mean_squared_error(y_tr_r, p_tr))
        va_mse.append(mean_squared_error(y_val_r, p_va))
    gb_curves[(d, lr_rate)] = (tr_mse, va_mse)

    met = reg_metrics(y_val_r, pipe.predict(X_val_r))
    met["max_depth"]     = d
    met["learning_rate"] = lr_rate
    rows.append(met)
    print(f"  GB d={d} lr={lr_rate}: MAE={met['MAE']:.2f} MSE={met['MSE']:.2f} RMSE={met['RMSE']:.2f}  R2={met['R2']:.4f}")

gb_reg_df = pd.DataFrame(rows)[["max_depth", "learning_rate", "MAE", "MSE", "RMSE", "R2"]]
gb_reg_df.to_csv(os.path.join(RES, "reg_ablation_gb.csv"), index=False)

# Curbe de convergenta (staged_predict)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for (d, lr), (tr, va) in gb_curves.items():
    axes[0].plot(range(1, len(tr)+1), tr, label=f"d={d} lr={lr}")
    axes[1].plot(range(1, len(va)+1), va, label=f"d={d} lr={lr}")
axes[0].set_title("Train MSE (GB staged_predict)")
axes[1].set_title("Validation MSE (GB staged_predict)")
for ax in axes:
    ax.set_xlabel("numar arbori"); ax.set_ylabel("MSE"); ax.legend()
plt.suptitle("Curbe de convergenta GradientBoosting")
plt.tight_layout()
plt.savefig(os.path.join(FIG, "reg_lc_gb_staged_MSE.png"), bbox_inches="tight"); plt.close()
print("[fig] reg_lc_gb_staged.png")

Ablatia 4: Gradient Boosting (n_estimators=1000)
  GB d=4 lr=0.1: MAE=4337.47 MSE=29521521.68 RMSE=5433.37  R2=0.9788
  GB d=5 lr=0.1: MAE=4394.38 MSE=30335629.79 RMSE=5507.78  R2=0.9782
[fig] reg_lc_gb_staged.png


Ablatia 5: Gradient Boosting (n_estimators=400, staged_predict)
  GB d=4 lr=0.1: MAE=4375.16  RMSE=5484.93  R2=0.9784
  GB d=5 lr=0.1: MAE=4420.97  RMSE=5554.43  R2=0.9778
[fig] reg_lc_gb_staged.png

In [60]:
# Curbe de invatare (MSE)
# LinearRegression vs GradientBoosting
print('Curbe de invatare (MSE)')

def mse_scorer(estimator, X, y):
    return -mean_squared_error(y, estimator.predict(X))

lc_frames = {}
models_lc = [
    ('LinearRegression',
     Pipeline([('prep', make_ct(True)), ('reg', LinearRegression())])),
    ('GradientBoosting',
     Pipeline([('prep', make_ct(False)),
               ('reg', GradientBoostingRegressor(
                   max_depth=4, learning_rate=0.1,
                   n_estimators=1000, random_state=RNG, subsample=0.8))])),
]

for name, pipe in models_lc:
    sizes = np.linspace(0.1, 1.0, 8)
    ts, tr_mse_s, va_mse_s = learning_curve(
        pipe, X_tr_r, y_tr_r, train_sizes=sizes, cv=3,
        scoring=mse_scorer, n_jobs=-1, random_state=RNG, shuffle=True)

    tr_mse = -tr_mse_s.mean(axis=1)
    va_mse = -va_mse_s.mean(axis=1)

    df_lc = pd.DataFrame({'n_samples': ts, 'train_MSE': tr_mse, 'val_MSE': va_mse})
    lc_frames[name] = df_lc
    df_lc.to_csv(os.path.join(RES, f'reg_learning_curve_{name}.csv'), index=False)
    print(f'  {name}: train_MSE={tr_mse[-1]:.0f}  val_MSE={va_mse[-1]:.0f}')

# Per-model: Train MSE vs Test MSE pe acelasi grafic (un grafic per model)
for name, df_lc in lc_frames.items():
    plt.figure(figsize=(7, 5))
    plt.plot(df_lc['n_samples'], df_lc['train_MSE'] / 1e6, color='green', marker='o', label='Train MSE w/ reg')
    plt.plot(df_lc['n_samples'], df_lc['val_MSE']   / 1e6, color='red',   marker='o', label='Test MSE w/ reg')
    plt.xlabel('M')
    plt.ylabel('MSE')
    plt.title(f'Comparatie pe setul de antrenare vs cel de\nvalidare pentru acelasi experiment ({name})')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG, f'reg_lc_{name}.png'), bbox_inches='tight')
    plt.close()
print('[fig] reg_lc_*.png (MSE train vs val per model) salvate')

Curbe de invatare (MSE)
  LinearRegression: train_MSE=66695091  val_MSE=66806585
  GradientBoosting: train_MSE=23812744  val_MSE=30291999
[fig] reg_lc_*.png (MSE train vs val per model) salvate


In [61]:
# Ridge, Lasso, LinearRegression, GradientBoosting
def best_row_r(df, metric='MAE', higher_better=False):
    return df.sort_values(metric, ascending=not higher_better).iloc[0].to_dict()

best_r = {
    'LinearRegression': base_val,
    'Ridge':            best_row_r(ridge_df),
    'Lasso':            best_row_r(lasso_df),
    'GradientBoosting': best_row_r(gb_reg_df),
}

def build_reg(name):
    if name == 'LinearRegression':
        return Pipeline([('prep', make_ct(True)), ('reg', LinearRegression())])
    if name == 'Ridge':
        return Pipeline([('prep', make_ct(True)),
                         ('reg', Ridge(alpha=float(best_r['Ridge']['alpha']),
                                        max_iter=10000, tol=1e-4, random_state=RNG))])
    if name == 'Lasso':
        return Pipeline([('prep', make_ct(True)),
                         ('reg', Lasso(alpha=float(best_r['Lasso']['alpha']),
                                        max_iter=10000, tol=1e-6, random_state=RNG))])

    if name == 'GradientBoosting':
        cfg = best_r['GradientBoosting']
        return Pipeline([('prep', make_ct(False)),
                         ('reg', GradientBoostingRegressor(
                             max_depth=int(cfg['max_depth']),
                             learning_rate=float(cfg['learning_rate']),
                             n_estimators=1000, random_state=RNG, subsample=0.8))])
    raise ValueError(name)

summary_val_r, summary_test_r = [], []
best_overall_name_r, best_overall_mae = None, float('inf')

for name in ['LinearRegression', 'Ridge', 'Lasso', 'GradientBoosting']:
    pipe = build_reg(name)
    pipe.fit(X_tr_r, y_tr_r)
    mv = reg_metrics(y_val_r, pipe.predict(X_val_r)); mv['model'] = name
    mt = reg_metrics(y_te_r,  pipe.predict(X_te_r));  mt['model'] = name
    summary_val_r.append(mv); summary_test_r.append(mt)
    if mv['MAE'] < best_overall_mae:
        best_overall_mae = mv['MAE']; best_overall_name_r = name

summary_val_r_df  = pd.DataFrame(summary_val_r)[['model','MAE','MSE','RMSE','R2']].sort_values('MAE')
summary_test_r_df = pd.DataFrame(summary_test_r)[['model','MAE','MSE','RMSE','R2']].sort_values('MAE')
summary_val_r_df.to_csv(os.path.join(RES, 'reg_summary_val.csv'), index=False)
summary_test_r_df.to_csv(os.path.join(RES, 'reg_summary_test.csv'), index=False)

with open(os.path.join(RES, 'reg_best_config.json'), 'w') as f:
    json.dump({'best_overall': best_overall_name_r,
               'best_params': {k: v for k, v in best_r.items()}},
              f, indent=2, default=str)

print('Sumar regresie pe validare')
print(summary_val_r_df.to_string(index=False))
print('Sumar regresie pe test')
print(summary_test_r_df.to_string(index=False))
print(f'Cel mai bun model: {best_overall_name_r} (val MAE = {best_overall_mae:.2f})')

#Grafice comparative MAE + MSE + RMSE (validare vs test)
models_order = summary_val_r_df['model'].tolist()
mae_val   = summary_val_r_df.set_index('model').loc[models_order, 'MAE'].values
mse_val   = summary_val_r_df.set_index('model').loc[models_order, 'MSE'].values
rmse_val  = summary_val_r_df.set_index('model').loc[models_order, 'RMSE'].values
mae_te    = summary_test_r_df.set_index('model').loc[models_order, 'MAE'].values
mse_te    = summary_test_r_df.set_index('model').loc[models_order, 'MSE'].values
rmse_te   = summary_test_r_df.set_index('model').loc[models_order, 'RMSE'].values

x = np.arange(len(models_order))
width = 0.4

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MAE
axes[0].bar(x - width/2, mae_val, width, label='Validare', color='steelblue')
axes[0].bar(x + width/2, mae_te,  width, label='Test',     color='coral')
axes[0].set_xticks(x); axes[0].set_xticklabels(models_order, rotation=20, ha='right')
axes[0].set_ylabel('MAE'); axes[0].set_title('Comparatie MAE')
axes[0].legend()
for i, (v, t) in enumerate(zip(mae_val, mae_te)):
    axes[0].text(i - width/2, v+30, f'{v:.0f}', ha='center', va='bottom', fontsize=7)
    axes[0].text(i + width/2, t+30, f'{t:.0f}', ha='center', va='bottom', fontsize=7)

# MSE
axes[1].bar(x - width/2, mse_val/1e6, width, label='Validare', color='steelblue')
axes[1].bar(x + width/2, mse_te/1e6,  width, label='Test',     color='coral')
axes[1].set_xticks(x); axes[1].set_xticklabels(models_order, rotation=20, ha='right')
axes[1].set_ylabel('MSE (milioane)'); axes[1].set_title('Comparatie MSE')
axes[1].legend()
for i, (v, t) in enumerate(zip(mse_val/1e6, mse_te/1e6)):
    axes[1].text(i - width/2, v+0.5, f'{v:.1f}M', ha='center', va='bottom', fontsize=7)
    axes[1].text(i + width/2, t+0.5, f'{t:.1f}M', ha='center', va='bottom', fontsize=7)

# RMSE
axes[2].bar(x - width/2, rmse_val, width, label='Validare', color='steelblue')
axes[2].bar(x + width/2, rmse_te,  width, label='Test',     color='coral')
axes[2].set_xticks(x); axes[2].set_xticklabels(models_order, rotation=20, ha='right')
axes[2].set_ylabel('RMSE'); axes[2].set_title('Comparatie RMSE')
axes[2].legend()
for i, (v, t) in enumerate(zip(rmse_val, rmse_te)):
    axes[2].text(i - width/2, v+30, f'{v:.0f}', ha='center', va='bottom', fontsize=7)
    axes[2].text(i + width/2, t+30, f'{t:.0f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Regresie: MAE, MSE si RMSE (validare vs test)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'reg_summary_mae_mse_rmse.png'), bbox_inches='tight'); plt.close()
print('[fig] reg_summary_mae_mse_rmse.png')

#Grafic dedicat MSE: validare vs test pentru fiecare model
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, mse_val / 1e6, width, label='Validare', color='steelblue')
ax.bar(x + width/2, mse_te  / 1e6, width, label='Test',     color='coral')
ax.set_xticks(x)
ax.set_xticklabels(models_order, rotation=20, ha='right')
ax.set_ylabel('MSE (milioane)')
ax.set_title('MSE per model: Validare vs Test')
ax.legend()
for i, (v, t) in enumerate(zip(mse_val / 1e6, mse_te / 1e6)):
    ax.text(i - width/2, v + 0.3, f'{v:.1f}M', ha='center', va='bottom', fontsize=8)
    ax.text(i + width/2, t + 0.3, f'{t:.1f}M', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'reg_summary_mse_only.png'), bbox_inches='tight'); plt.close()
print('[fig] reg_summary_mse_only.png')

#Grafic MSE per model: Train vs Test (reantrenat pe X_tr_r, evaluat pe amandoua)
mse_tr_list = []
for name in models_order:
    pipe = build_reg(name)
    pipe.fit(X_tr_r, y_tr_r)
    mse_tr_list.append(mean_squared_error(y_tr_r, pipe.predict(X_tr_r)))
mse_tr_arr = np.array(mse_tr_list)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, mse_tr_arr / 1e6, width, label='Train', color='green')
ax.bar(x + width/2, mse_val    / 1e6, width, label='Test',  color='red')
ax.set_xticks(x)
ax.set_xticklabels(models_order, rotation=20, ha='right')
ax.set_ylabel('MSE (milioane)')
ax.set_title('MSE per model: Train vs Test')
ax.legend()
for i, (tr, te) in enumerate(zip(mse_tr_arr / 1e6, mse_val / 1e6)):
    ax.text(i - width/2, tr + 0.3, f'{tr:.1f}M', ha='center', va='bottom', fontsize=8)
    ax.text(i + width/2, te + 0.3, f'{te:.1f}M', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'reg_summary_mse_train_test.png'), bbox_inches='tight'); plt.close()
print('[fig] reg_summary_mse_train_test.png')


### Sumar regresie pe val ###
           model         MAE          MSE        RMSE       R2
GradientBoosting 4337.466326 2.952152e+07 5433.371116 0.978759
           Lasso 6303.830741 6.758606e+07 8221.074161 0.951370
           Ridge 6305.775947 6.759930e+07 8221.879407 0.951361
LinearRegression 6305.966880 6.760015e+07 8221.930913 0.951360
### Sumar regresie pe test ###
           model         MAE          MSE        RMSE       R2
GradientBoosting 4345.490430 2.970460e+07 5450.192265 0.978496
           Lasso 6301.066967 6.556214e+07 8097.045257 0.952539
           Ridge 6302.591501 6.556183e+07 8097.025795 0.952539
LinearRegression 6303.006495 6.555930e+07 8096.870072 0.952541
Cel mai bun model: GradientBoosting (val MAE = 4337.47)
[fig] reg_summary_mae_mse_rmse.png
[fig] reg_summary_mse_only.png
[fig] reg_summary_mse_train_test.png


In [57]:
# KAGGLE REGRESSION
import os
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor

KAGGLE_REG_PATH = os.path.join(os.path.dirname(ROOT), "CC_private_test_regression.csv")
# antrenez pe toate datele
all_labeled_reg = pd.concat([df, df_test], ignore_index=True)
df_private_reg  = pd.read_csv(KAGGLE_REG_PATH)

print(f"Antrenam pe {len(all_labeled_reg)} instante (putere maxima!)")

# Adaug interatiuni intre feature-uri corelate
def add_interactions_reg(d):
    edu_map  = {v: i for i, v in enumerate(EDU_ORDER)}   
    comp_map = {v: i for i, v in enumerate(COMP_ORDER)}  
    d = d.copy()
    d["edu_rank"]       = d["education_level"].map(edu_map).fillna(2.0)
    d["comp_rank"]      = d["company_size"].map(comp_map).fillna(2.0)
    d["edu_comp_score"] = d["edu_rank"] * d["comp_rank"]  
    d["exp_edu_score"]  = d["experience_years"].fillna(10.0) * d["edu_rank"]
    return d

y_all_reg = all_labeled_reg["salary"]
X_all_raw_reg  = all_labeled_reg.drop(columns=["vacation", "salary"])
X_priv_raw_reg = df_private_reg.drop(columns=["vacation", "salary"], errors="ignore")

X_all_aug_reg  = add_interactions_reg(X_all_raw_reg)
X_priv_aug_reg = add_interactions_reg(X_priv_raw_reg)

# Pipeline ca la clasificare
NUM_COLS = ["experience_years", "skills_count", "certifications",
            "edu_rank", "comp_rank", "edu_comp_score", "exp_edu_score"]
ORD_COLS = ["skill_bracket"]
OHE_COLS = ["education_level", "company_size", "job_title",
            "industry", "location", "remote_work"]

num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler())
])
ord_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("enc", OrdinalEncoder(
        categories=[BRACKET_ORDER],
        handle_unknown="use_encoded_value", unknown_value=-1))
])
ohe_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

ct_final_reg = ColumnTransformer([
    ("num", num_pipe, NUM_COLS),
    ("ord", ord_pipe, ORD_COLS),
    ("ohe", ohe_pipe, OHE_COLS),
], remainder="drop")

# GradientBoostingRegressor
reg_final = GradientBoostingRegressor(
    loss='squared_error',
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=RNG
)
print("Preprocesare")
X_train_final_reg  = ct_final_reg.fit_transform(X_all_aug_reg)
X_kaggle_final_reg = ct_final_reg.transform(X_priv_aug_reg)

print("Antrenare")
reg_final.fit(X_train_final_reg, y_all_reg)

# Predictie
print("Predictie")
preds_reg = reg_final.predict(X_kaggle_final_reg)

submission_reg = pd.DataFrame({"id": df_private_reg["id"], "prediction": preds_reg})
submission_reg.to_csv("submission_regression_v2.csv", index=False)


Antrenam pe 80000 instante (putere maxima!)
Preprocesez datele...
Antrenez algoritmul...
Generez predictiile...
GATA! Am salvat 16000 predictii de salariu in 'submission_GB_regresie_v2.csv'


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor

KAGGLE_REG_PATH = os.path.join(os.path.dirname(ROOT), "CC_private_test_regression.csv")

all_labeled_reg = pd.concat([df, df_test], ignore_index=True)
df_private_reg  = pd.read_csv(KAGGLE_REG_PATH)

X_all_raw  = all_labeled_reg.drop(columns=["vacation", "salary"])
y_all_reg  = all_labeled_reg["salary"]
X_priv_raw = df_private_reg.drop(columns=["vacation", "salary"], errors="ignore")

MIN_SALARY = y_all_reg.min()
MAX_SALARY = y_all_reg.max()

# Pipeline curat (fara feature engineering)
NUM_COLS = ["experience_years", "skills_count", "certifications"]
ORD_COLS = ["skill_bracket", "education_level", "company_size"]
OHE_COLS = ["job_title", "industry", "location", "remote_work"]

num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler())
])
ord_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("enc", OrdinalEncoder(
        categories=[BRACKET_ORDER, EDU_ORDER, COMP_ORDER],
        handle_unknown="use_encoded_value", unknown_value=-1))
])
ohe_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

ct_final_reg = ColumnTransformer([
    ("num", num_pipe, NUM_COLS),
    ("ord", ord_pipe, ORD_COLS),
    ("ohe", ohe_pipe, OHE_COLS),
], remainder="drop")

reg_final = GradientBoostingRegressor(
    loss='squared_error',
    n_estimators=1000,
    max_depth=3,
    min_samples_leaf=20,
    learning_rate=0.05,
    subsample=0.8,
    random_state=RNG
)
# aveam n_estimators=800, min_samples_leaf=15
# apoi 1000 cu 20 - 720, lr = 0.05
# apoi 1500 cu 20 cu lr = 0.03
print("Preprocesare + antrenare")
X_train_final = ct_final_reg.fit_transform(X_all_raw)
X_kaggle_final = ct_final_reg.transform(X_priv_raw)

reg_final.fit(X_train_final, y_all_reg)

preds_raw = reg_final.predict(X_kaggle_final)

preds_clipped = np.clip(preds_raw, MIN_SALARY, MAX_SALARY)

submission_reg = pd.DataFrame({"id": df_private_reg["id"], "prediction": preds_clipped})
submission_reg.to_csv("submission_regression_v1.csv", index=False)

